# Онлайн-деканат — запуск в Google Colab

Ноутбук устанавливает зависимости, прогоняет открытый набор через конвейер (OCR → поля → проверка → ответы) и считает метрики `evaluate.py`. Внешние API не используются: все модели работают локально (CPU).

In [ ]:
!git clone https://github.com/KochetkovAlEX/umirhack.git
%cd umirhack
!apt-get -qq install -y tesseract-ocr tesseract-ocr-rus > /dev/null
!pip -q install -r requirements/core.txt

In [ ]:
!python scripts/download_models.py

In [ ]:
# Открытый набор: PNG -> results.json + метрики
!python scripts/predict.py --manifest data/documents.json --questions data/questions.json --base data --input image --out results.json --reference data/reference_results.json

In [ ]:
# Один документ и вопрос «вручную»
import sys; sys.path.insert(0, 'core')
from dekanat_core import pipeline
res, ctx = pipeline.process_document(open('data/documents/open-002.png','rb').read(), 'open-002.png', None)
print(res.doc_type, res.type_confidence)
print(res.fields_plain())
print(res.validation)
for q in ['Какова дата выдачи?', 'Кто принимает окончательное решение?', 'Какой размер стипендии?']:
    a = pipeline.answer_question(res, q); print(q, '->', a.answerable, a.value, a.sources, '|', a.answer)

In [ ]:
# Закрытый набор (положите файлы в data/closed/ с documents.json и questions.json)
# !python scripts/predict.py --manifest data/closed/documents.json --questions data/closed/questions.json --base data/closed --input image --out results.json